<a href="https://colab.research.google.com/github/ssk-algoverse/sae-binding/blob/main/circuit/Exploratory%20analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformer_lens

In [2]:
import torch
from huggingface_hub import hf_hub_download
from transformer_lens import HookedTransformer, HookedTransformerConfig
import numpy as np
import pandas as pd
import ast
from torch.utils.data import Dataset, DataLoader

In [5]:
from huggingface_hub import hf_hub_download

REPO_ID = "sebastianhoenig/2L_1H_No_Bias_Entity_Binding" # "sebastianhoenig/2L_1H_Entity_Binding"
FILENAME = "entity_binding_model2.pth" # "2L_1H_Attn_Only.pth"

weights_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)

entity_binding_model2.pth:   0%|          | 0.00/2.63M [00:00<?, ?B/s]

In [12]:
### Model


E = 100 # num entities
A = 100 # num attributes
T = 10 # num types/relations
SEP = E+A+T # as seperator between relations
Q = E+A+T+1 # question token
PAD = E+A+T+2
D_VOCAB = E+A+T+3
IGNORE_INDEX = -100

cfg = HookedTransformerConfig(
    n_layers=2,
    n_heads=1,
    d_model=256,
    d_head=256,
    n_ctx=64,
    d_vocab=D_VOCAB,
    act_fn="gelu",
    attn_only=True,
    normalization_type="LN",
    use_attn_result=True
)
model = HookedTransformer(cfg)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Load the model
# Create a new model instance with the same configuration
pretrained_weights = torch.load(weights_path, map_location=device, weights_only=True)
model.load_state_dict(pretrained_weights)

print("Model loaded successfully.")

Moving model to device:  cpu
Model loaded successfully.


In [13]:
id_mapping_df = pd.read_csv('id_mapping.csv')
id_to_entity = dict(zip(id_mapping_df['id'], id_mapping_df['name']))

In [14]:
class EntityBindingDataset(Dataset):
    def __init__(self, dataframe, parse_tokens_if_str=True):
        self.df = dataframe.reset_index(drop=True)
        self.parse_tokens_if_str = parse_tokens_if_str

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        seq = row["tokens"]
        tokens = torch.tensor(seq, dtype=torch.long)
        label  = torch.tensor(int(row["label"]), dtype=torch.long)
        return tokens, label

test_df = pd.read_csv('test_df.csv', converters={"tokens": ast.literal_eval})
test_dataset = EntityBindingDataset(test_df)

In [15]:
### Some very basic checks: What happens if we pertube the input sequence

In [16]:
id_to_entity[210] = ","
id_to_entity[211] = "?"
example, label = test_dataset[0]

In [17]:
print("Example tokens and their mapped entities:")
for token_id in example:
    entity_name = id_to_entity.get(token_id.item(), f"Unknown Token {token_id.item()}")
    print(f"Token {token_id.item()} ({entity_name})")

print(f"\nLabel: {label.item()} ({id_to_entity.get(label.item(), f'Unknown Token {label.item()}')})")

Example tokens and their mapped entities:
Token 94 (Bryan)
Token 201 (was born in)
Token 118 (Istanbul)
Token 210 (,)
Token 43 (Keith)
Token 208 (moved to)
Token 105 (Manila)
Token 210 (,)
Token 18 (Lisa)
Token 200 (lives in)
Token 155 (Philadelphia)
Token 210 (,)
Token 95 (Patrick)
Token 204 (studied in)
Token 144 (Riyadh)
Token 210 (,)
Token 48 (Cynthia)
Token 202 (works in)
Token 106 (Shanghai)
Token 210 (,)
Token 37 (Christine)
Token 205 (married in)
Token 111 (Sao Paulo)
Token 210 (,)
Token 54 (Deborah)
Token 206 (visited)
Token 169 (Cape Town)
Token 210 (,)
Token 80 (Anna)
Token 207 (loves)
Token 154 (Miami)
Token 210 (,)
Token 208 (moved to)
Token 43 (Keith)
Token 211 (?)

Label: 105 (Manila)


In [18]:
## considering the incorrect token to be the 10th most probable prediction
## take a batch of examples with this heuristic

In [19]:
try:
    import google.colab # type: ignore
    IN_COLAB = True
except:
    IN_COLAB = False

import os, sys
chapter = "chapter1_transformer_interp"
repo = "ARENA_3.0"

if IN_COLAB:
    # Install packages
    %pip install transformer_lens
    %pip install einops
    %pip install jaxtyping
    %pip install git+https://github.com/callummcdougall/CircuitsVis.git#subdirectory=python

    # Code to download the necessary files (e.g. solutions, test funcs)
    if not os.path.exists(f"/content/{chapter}"):
        !wget https://github.com/callummcdougall/ARENA_3.0/archive/refs/heads/main.zip
        !unzip /content/main.zip 'ARENA_3.0-main/chapter1_transformer_interp/exercises/*'
        sys.path.append(f"/content/{repo}-main/{chapter}/exercises")
        os.remove("/content/main.zip")
        os.rename(f"{repo}-main/{chapter}", chapter)
        os.rmdir(f"{repo}-main")
        os.chdir(f"{chapter}/exercises")
else:
    chapter_dir = r"./" if chapter in os.listdir() else os.getcwd().split(chapter)[0]
    sys.path.append(chapter_dir + f"{chapter}/exercises")

  Cloning https://github.com/callummcdougall/CircuitsVis.git to /tmp/pip-req-build-rmsnq6yd
  Running command git clone --filter=blob:none --quiet https://github.com/callummcdougall/CircuitsVis.git /tmp/pip-req-build-rmsnq6yd
  Resolved https://github.com/callummcdougall/CircuitsVis.git to commit 1e6129d08cae7af9242d9ab5d3ed322dd44b4dd3
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for circuitsvis: filename=circuitsvis-0.0.0-py3-none-any.whl size=6172337 sha256=a4f68a36d13a4fbe2ed1d8bde97d80050bd39c247f48ad5a9f309c1c97601ffd
  Stored in directory: /tmp/pip-ephem-wheel-cache-db2y47ix/wheels/00/ce/19/651aed367fa8cefad943dece40a2248cef6588697047472ef1
Successfully built circuitsvis
  Attempting uninstall: importlib-metadata
    Found existing installation: importlib-metadata 4.6.4
    Uninstalling importlib-metadata-4.6.4:
      Successfully uninstalled importlib-metadata-4.6.4
--2025-

In [20]:
from rich.table import Table, Column, box
from rich import print as rprint
from jaxtyping import Float, Int, Bool
from typing import List, Optional, Callable, Tuple, Dict, Literal, Set, Union
from torch import Tensor
from transformer_lens import ActivationCache
import einops
from rich import print as rprint
from plotly_utils import imshow, line, scatter, bar
from pathlib import Path
from IPython.display import display, HTML
from transformer_lens import utils
import circuitsvis as cv


In [21]:
import random
random.seed(42)
same_seq_subset = [ex for ex in test_dataset if len(ex[0]) == 23]

start, end = 0, len(same_seq_subset)
count = 10
random_ints = [random.randint(start, end) for _ in range(count)]

dla_dataset = [same_seq_subset[idx] for idx in random_ints]


In [22]:
text_examples = []
text_labels = []
text_incorrect_answers = []

incorrect_tokens = []
label_tokens = []
example_tokens = []

top1_probs = []
top10_probs = []

logit_diffs = []
acc = 0

for ex in dla_dataset:
  # get model prediction
  input, label = ex
  with torch.no_grad():
    logits = model(input)

  probs = logits[0, -1, :].softmax(dim=-1)
  top_values, top_indices = probs.topk(10)
  incorrect_token = top_indices.tolist()[-1]

  text_incorrect_answers.append(id_to_entity[incorrect_token])
  incorrect_tokens.append(incorrect_token)

  text_examples.append(" ".join([id_to_entity[tok] for tok in ex[0].tolist()]))
  example_tokens.append(ex)

  text_labels.append(id_to_entity[label.item()])
  label_tokens.append(label.item())

  top1_probs.append(top_values.tolist()[0])
  top10_probs.append(top_values.tolist()[-1])

  correct_token = top_indices.tolist()[0]
  if correct_token == label.item():
    acc += 1

  logit_diffs.append(logits[0, -1, label.item()] - logits[0, -1, incorrect_token])

In [23]:
cols = [
    "Prompt",
    Column("Correct", style="rgb(0,200,0) bold"),
    Column("Incorrect", style="rgb(255,0,0) bold"),
    Column("Logit Difference", style="bold")
]
table = Table(*cols, title="Logit differences", show_lines=True)

for prompt, answer, incorrect_answer, logit_diff in zip(text_examples, text_labels, text_incorrect_answers, logit_diffs):
    table.add_row(prompt, repr(answer), repr(incorrect_answer), f"{logit_diff.item():.3f}")

rprint(table)

                                                 Logit differences                                                 
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ Prompt                                                     ┃ Correct        ┃ Incorrect      ┃ Logit Difference ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Cassandra left Daejeon , Leonard works in Zhengzhou ,      │ 'Incheon'      │ 'Baghdad'      │ 21.883           │
│ Robert moved to Wuhan , Christopher lives in Incheon ,     │                │                │                  │
│ Keith loves Dallas , lives in Christopher ?                │                │                │                  │
├────────────────────────────────────────────────────────────┼────────────────┼────────────────┼──────────────────┤
│ William was born in Guangzhou , Matthew works in Pune ,    │ 'Guangzhou'    │ 'Yangon'       │ 20.321           │
│ Antonio studied in Chicago , Leonard moved to Mumbai ,     │                │                │                  │
│ Jessica visited Shenzhen , was born in William ?           │                │                │                  │
├────────────────────────────────────────────────────────────┼────────────────┼────────────────┼──────────────────┤
│ Matthew works in Wuhan , Allison lives in Chicago , Jeremy │ 'Philadelphia' │ 'Daegu'        │ 20.282           │
│ visited Philadelphia , Sharon was born in Cairo , Brenda   │                │                │                  │
│ moved to Karachi , visited Jeremy ?                        │                │                │                  │
├────────────────────────────────────────────────────────────┼────────────────┼────────────────┼──────────────────┤
│ Diana lives in Melbourne , Jeremy travels to Manila ,      │ 'Berlin'       │ 'Singapore'    │ 21.341           │
│ Christine visited Karachi , Bridget studied in Berlin ,    │                │                │                  │
│ Wendy was born in Baghdad , studied in Bridget ?           │                │                │                  │
├────────────────────────────────────────────────────────────┼────────────────┼────────────────┼──────────────────┤
│ Angela visited Osaka , Lisa lives in Kinshasa , Deborah    │ 'Osaka'        │ 'Casablanca'   │ 21.164           │
│ left Mumbai , Melanie studied in Qingdao , Anthony travels │                │                │                  │
│ to Sendai , visited Angela ?                               │                │                │                  │
├────────────────────────────────────────────────────────────┼────────────────┼────────────────┼──────────────────┤
│ Nicholas works in Zhengzhou , Matthew lives in Daegu ,     │ 'Zhengzhou'    │ 'Baghdad'      │ 20.388           │
│ Tasha loves Santiago , Teresa studied in Baghdad ,         │                │                │                  │
│ Patricia left Philadelphia , works in Nicholas ?           │                │                │                  │
├────────────────────────────────────────────────────────────┼────────────────┼────────────────┼──────────────────┤
│ George studied in Guangzhou , Christopher married in New   │ 'Shenzhen'     │ 'Wuhan'        │ 20.455           │
│ York , Kimberly travels to Incheon , Janet visited         │                │                │                  │
│ Shenzhen , Robert was born in Kano , visited Janet ?       │                │                │                  │
├────────────────────────────────────────────────────────────┼────────────────┼────────────────┼──────────────────┤
│ Michelle studied in Delhi , Michael loves Shenzhen , Tasha │ 'Ulsan'        │ 'Porto Alegre' │ 15.541           │
│ left Yokohama , Keith travels to Dallas , Jeffrey moved to │                │                │                  │
│ Ulsan , moved to Jeffrey ?                            

In [24]:
## logit lens

In [25]:
def residual_stack_to_logit_diff(
    residual_stack: Float[Tensor, "... batch d_model"], # contains residual stream values for the final sequence position
    cache: ActivationCache,
    logit_diff_directions: Float[Tensor, "batch d_model"],
) -> Float[Tensor, "..."]:
    '''
    Gets the avg logit difference between the correct and incorrect answer for a given
    stack of components in the residual stream.
    '''
    print(residual_stack.shape)
    ln_residual_stack = cache.apply_ln_to_stack(residual_stack, layer=-1, pos_slice=-1)
    average_logit_diff = einops.einsum(ln_residual_stack, logit_diff_directions, "... batch d_model, batch d_model ->...") / residual_stack.shape[0]
    return average_logit_diff

In [26]:
dla_dataset = torch.stack([ex[0] for ex in dla_dataset], dim=0)
dla_dataset.shape

torch.Size([10, 23])

In [27]:
# this is equivalent to indexing the unembedding matrix and getting the column corresponding to the given index/token
answer_token_directions = model.tokens_to_residual_directions(torch.tensor(label_tokens))
incorrect_token_directions = model.tokens_to_residual_directions(torch.tensor(incorrect_tokens))
logit_diff_directions = answer_token_directions - incorrect_token_directions


In [28]:
# verification of the method
(answer_token_directions[0] == model.W_U[:, label_tokens[0]]).sum() == 256

tensor(True)

In [29]:
original_logits, cache = model.run_with_cache(dla_dataset)

In [30]:
accumulated_residual, labels = cache.accumulated_resid(layer=-1, pos_slice=-1, return_labels=True)
# accumulated_residual has shape (component, batch, d_model)
# 12 blocks, so 12 attn, 12 mlp and one input layer
print("acc", accumulated_residual.shape)
logit_lens_logit_diffs: Float[Tensor, "component"] = residual_stack_to_logit_diff(accumulated_residual, cache, logit_diff_directions)

line(
    logit_lens_logit_diffs,
    hovermode="x unified",
    title="Logit Difference From Accumulated Residual Stream",
    labels={"x": "Layer", "y": "Logit Diff"},
    xaxis_tickvals=labels,
    width=800
)

acc torch.Size([3, 10, 256])
torch.Size([3, 10, 256])


In [31]:
per_layer_residual, labels = cache.decompose_resid(layer=-1, pos_slice=-1, return_labels=True)
per_layer_logit_diffs = residual_stack_to_logit_diff(per_layer_residual, cache, logit_diff_directions)

line(
    per_layer_logit_diffs,
    hovermode="x unified",
    title="Logit Difference From Each Layer",
    labels={"x": "Layer", "y": "Logit Diff"},
    xaxis_tickvals=labels,
    width=800
)

torch.Size([4, 10, 256])


In [32]:
per_head_residual, labels = cache.stack_head_results(layer=-1, pos_slice=-1, return_labels=True)
per_head_residual = einops.rearrange(
    per_head_residual,
    "(layer head) ... -> layer head ...",
    layer=model.cfg.n_layers
)
per_head_logit_diffs = residual_stack_to_logit_diff(per_head_residual, cache, logit_diff_directions)

imshow(
    per_head_logit_diffs,
    labels={"x":"Head", "y":"Layer"},
    title="Logit Difference From Each Head",
    width=600
)

torch.Size([2, 1, 10, 256])


In [33]:
def topk_of_Nd_tensor(tensor: Float[Tensor, "rows cols"], k: int):
    '''
    Helper function: does same as tensor.topk(k).indices, but works over 2D tensors.
    Returns a list of indices, i.e. shape [k, tensor.ndim].

    Example: if tensor is 2D array of values for each head in each layer, this will
    return a list of heads.
    '''
    i = torch.topk(tensor.flatten(), k).indices
    return np.array(np.unravel_index(utils.to_numpy(i), tensor.shape)).T.tolist()


k = 2

for head_type in ["Positive", "Negative"]:

    # Get the heads with largest (or smallest) contribution to the logit difference
    top_heads = topk_of_Nd_tensor(per_head_logit_diffs * (1 if head_type=="Positive" else -1), k)

    # Get all their attention patterns
    attn_patterns_for_important_heads: Float[Tensor, "head q k"] = torch.stack([
        cache["pattern", layer][:, head].mean(0)
         for layer, head in top_heads
    ])

    # Display results
    display(HTML(f"<h2>Top {k} {head_type} Logit Attribution Heads</h2>"))
    display(cv.attention.attention_patterns(
        attention = attn_patterns_for_important_heads,
        tokens = [id_to_entity[tok] for tok in dla_dataset[4].tolist()],
        attention_head_names = [f"{layer}.{head}" for layer, head in top_heads],
    ))

In [34]:
clean = dla_dataset[4]
clean, " ".join([id_to_entity[tok] for tok in clean.tolist()])

(tensor([ 62, 206, 188, 210,  18, 200, 122, 210,  54, 209, 104, 210,  28, 204,
         165, 210,  12, 203, 191, 210, 206,  62, 211]),
 'Angela visited Osaka , Lisa lives in Kinshasa , Deborah left Mumbai , Melanie studied in Qingdao , Anthony travels to Sendai , visited Angela ?')

In [35]:
example_tokens[0], text_examples[0]

((tensor([ 29, 209, 195, 210,  73, 202, 166, 210,  93, 208, 136, 210,  88, 200,
          198, 210,  43, 207, 152, 210, 200,  88, 211]),
  tensor(198)),
 'Cassandra left Daejeon , Leonard works in Zhengzhou , Robert moved to Wuhan , Christopher lives in Incheon , Keith loves Dallas , lives in Christopher ?')

In [36]:
corrupt_entity = 29 # Cassandra
corrupt_relation = 202 # works in
ic_relation = 200 # lives in
ic_entity = 18 # Lisa

In [37]:
## clean
print(" ".join([id_to_entity[tok] for tok in clean.tolist()]))
## both entity and relation are not in context
corrupt_no_entity_no_rel = clean.tolist()[:-3] + [corrupt_relation, corrupt_entity, 211]
print(" ".join([id_to_entity[tok] for tok in corrupt_no_entity_no_rel]))
## entity is there but relation not in context
corrupt_no_rel = clean.tolist()[:-3] + [corrupt_relation, ic_entity, 211]
print(" ".join([id_to_entity[tok] for tok in corrupt_no_rel]))
## relation is there but entity not in context
corrupt_no_entity = clean.tolist()[:-3] + [ic_relation, corrupt_entity, 211]
print(" ".join([id_to_entity[tok] for tok in corrupt_no_entity]))
## both entity and relaion are in context, but not binded
corrupt_both_in_context = clean.tolist()[:-2] + [ic_entity, 211]
print(" ".join([id_to_entity[tok] for tok in corrupt_both_in_context]))

Angela visited Osaka , Lisa lives in Kinshasa , Deborah left Mumbai , Melanie studied in Qingdao , Anthony travels to Sendai , visited Angela ?
Angela visited Osaka , Lisa lives in Kinshasa , Deborah left Mumbai , Melanie studied in Qingdao , Anthony travels to Sendai , works in Cassandra ?
Angela visited Osaka , Lisa lives in Kinshasa , Deborah left Mumbai , Melanie studied in Qingdao , Anthony travels to Sendai , works in Lisa ?
Angela visited Osaka , Lisa lives in Kinshasa , Deborah left Mumbai , Melanie studied in Qingdao , Anthony travels to Sendai , lives in Cassandra ?
Angela visited Osaka , Lisa lives in Kinshasa , Deborah left Mumbai , Melanie studied in Qingdao , Anthony travels to Sendai , visited Lisa ?


In [38]:
def print_stats(logits):
  probs = logits[0, -1, :].softmax(dim=-1)*100
  top_values, top_indices = probs.topk(10)
  top_values, top_indices = top_values.tolist(), top_indices.tolist()
  for idx in range(10):
    print(f"{top_indices[idx]}, {id_to_entity[top_indices[idx]]}, {top_values[idx]}%")

In [39]:
clean_logits = model(clean)
corrupt1_logits = model(torch.tensor(corrupt_no_entity_no_rel))
corrupt2_logits = model(torch.tensor(corrupt_no_rel))
corrupt3_logits = model(torch.tensor(corrupt_no_entity))
corrupt4_logits = model(torch.tensor(corrupt_both_in_context))

In [40]:
print("Clean")
print_stats(clean_logits)
print("=================================================")
print("No entity no relation in context")
print_stats(corrupt1_logits)
print("==================================================")
print("No relation in context")
print_stats(corrupt2_logits)
print("==================================================")
print("No entity in context")
print_stats(corrupt3_logits)
print("==================================================")
print("No entity entity_type binding in context")
print_stats(corrupt4_logits)

Clean
188, Osaka, 99.76840209960938%
122, Kinshasa, 0.2315923571586609%
191, Sendai, 1.1399514505683328e-06%
184, Addis Ababa, 5.204603894526372e-07%
117, Buenos Aires, 3.266682142566424e-07%
110, Kolkata, 2.968918124679476e-07%
102, Delhi, 2.163605472560448e-07%
146, Santiago, 7.378530852975018e-08%
173, Brasilia, 6.4413796962981e-08%
172, Casablanca, 6.417735676222946e-08%
No entity no relation in context
122, Kinshasa, 96.68634796142578%
191, Sendai, 3.2290403842926025%
188, Osaka, 0.04645473510026932%
165, Qingdao, 0.027157139033079147%
104, Mumbai, 0.010447229258716106%
102, Delhi, 9.119284368352965e-05%
110, Kolkata, 4.442219142219983e-05%
156, Atlanta, 2.8646922146435827e-05%
131, London, 2.7004207368008792e-05%
103, Guangzhou, 2.380691330472473e-05%
No relation in context
122, Kinshasa, 83.36148834228516%
191, Sendai, 16.618324279785156%
188, Osaka, 0.013448773883283138%
165, Qingdao, 0.005397292319685221%
104, Mumbai, 0.0006494127446785569%
102, Delhi, 9.398395923199132e-05%
1

In [49]:
relation, entity, question = clean.tolist()[-3:]
attribute = 188

predict_relation = clean.tolist()[:-3] + [attribute, entity, question]
print(" ".join([id_to_entity[tok] for tok in predict_relation]))

Angela visited Osaka , Lisa lives in Kinshasa , Deborah left Mumbai , Melanie studied in Qingdao , Anthony travels to Sendai , Osaka Angela ?


In [46]:
predict_entity = clean.tolist()[:-3] + [relation, attribute, question]
print(" ".join([id_to_entity[tok] for tok in predict_entity]))


Angela visited Osaka , Lisa lives in Kinshasa , Deborah left Mumbai , Melanie studied in Qingdao , Anthony travels to Sendai , visited Osaka ?


In [47]:
pred_ent_logits = model(torch.tensor(predict_entity))
print_stats(pred_ent_logits)

122, Kinshasa, 75.5372314453125%
188, Osaka, 24.292116165161133%
191, Sendai, 0.08945975452661514%
104, Mumbai, 0.07983066886663437%
117, Buenos Aires, 0.0002687683154363185%
102, Delhi, 0.00018360595277044922%
184, Addis Ababa, 8.991156209958717e-05%
172, Casablanca, 5.601925295195542e-05%
120, Rio de Janeiro, 5.355415851227008e-05%
156, Atlanta, 5.045361467637122e-05%


In [48]:
pred_rel_logits = model(torch.tensor(predict_relation))
print_stats(pred_rel_logits)

188, Osaka, 99.82605743408203%
122, Kinshasa, 0.173948273062706%
191, Sendai, 9.157127465186932e-07%
184, Addis Ababa, 4.5451406549545936e-07%
117, Buenos Aires, 2.644445658006589e-07%
110, Kolkata, 2.5402877668057045e-07%
102, Delhi, 1.913990956836642e-07%
146, Santiago, 6.164110288864322e-08%
172, Casablanca, 5.4453725084613325e-08%
173, Brasilia, 5.3176883341166103e-08%
